# 필요 라이브러리 import

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
from sklearn.metrics import silhouette_score
import seaborn as sns
import os  
from openai import AzureOpenAI  
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from xgboost import XGBRegressor
import shap

In [ ]:
api_key = 'YOUR_API_KEY'


# 1. ETF 지표 기반 군집화 후 군집 해석 및 군집별 대표 ETF 큐레이션

## ETF점수정보 테이블의 ETF 지표를 이용해서 군집화 진행

In [ ]:
# ETF점수정보 테이블 불러오기
ETF_SOR_IFO = pd.read_csv('./data/NH_CONTEST_ETF_SOR_IFO.csv', encoding='cp949')

# 종목 코드 뒤 공백 제거
ETF_SOR_IFO['etf_iem_cd'] = ETF_SOR_IFO['etf_iem_cd'].str.rstrip()

# 날짜 순 정렬
ETF_SOR_IFO.sort_values(by=['etf_iem_cd', 'bse_dt'], ascending=True, inplace=True)
ETF_SOR_IFO

- ETF점수정보 테이블에서 총 7개의 ETF 지표(누적수익율, 정보비율, 샤프지수, 상관관계, 트래킹에러, 최대낙폭, 변동성)를 사용합니다.
- 3개월(2024.05.28~2024.08.27)간의 데이터의 평균을 구해서 각 ETF의 최종 지표로 사용합니다.

In [ ]:
# ETF점수정보 테이블을 이용하여 ETF별 평균 점수를 계산
etf_score = ETF_SOR_IFO.groupby('etf_iem_cd').mean()
etf_score = etf_score.drop(columns=['bse_dt', 'mm1_tot_pft_rt', 'mm3_tot_pft_rt', 'yr1_tot_pft_rt', 'etf_sor', 'etf_z_sor', 'z_sor_rnk'])
etf_score.head()

- 군집화 방법은 K-means 클러스터링으로 선택했습니다.
- 최적의 군집 개수를 구하기 위해 엘보우 기법(Elbow method)을 사용했습니다.
- 그 결과, 최적의 클러스터 개수를 3개로 지정했습니다.

In [ ]:
# \엘보우 기법을 적용해 K-means 클러스터링 최적의 군집 수 찾기
range_n_clusters = range(2, 11)

sse = []
silhouette_scores = []

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(etf_score)
    
    # 엘보우 기법을 위한 SSE 저장
    sse.append(kmeans.inertia_)
    
    # 실루엣 점수 계산
    silhouette_avg = silhouette_score(etf_score, kmeans.labels_)
    silhouette_scores.append(silhouette_avg)

# 결과 시각화 (엘보우 기법)
plt.figure(figsize=(6, 5))

plt.plot(range_n_clusters, sse, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of clusters')
plt.ylabel('SSE')

plt.tight_layout()
plt.show()

- 클러스터링 결과를 시각화한 것은 아래와 같습니다.

In [ ]:
# 최적의 클러스터 개수 선택
optimal_n_clusters = 3

# K-means 클러스터링 수행
kmeans = KMeans(n_clusters=optimal_n_clusters, random_state=42)
kmeans.fit(etf_score)

# 각 데이터 포인트의 클러스터 레이블 가져오기
labels = kmeans.labels_

# 차원 축소 (PCA를 사용하여 2차원으로 차원축소)
pca = PCA(n_components=2)
data_2d = pca.fit_transform(etf_score)

# 클러스터링 결과 시각화 (2차원 데이터)
plt.scatter(data_2d[:, 0], data_2d[:, 1], c=labels, cmap='viridis')
# plt.scatter(pca.transform(kmeans.cluster_centers_)[:, 0], pca.transform(kmeans.cluster_centers_)[:, 1], s=300, c='red', label='Centroids')
plt.title(f'K-means Clustering with {optimal_n_clusters} Clusters (2D PCA)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

##  군집화 결과 해석에 기반하여, 각 군집의 이름을 명명

In [ ]:
# 군집별 평균값 확인
etf_score['cluster'] = labels
etf_score.groupby('cluster').mean()

In [ ]:
# 군집 해석을 위한 컬럼 별 boxplot 시각화

metrics = etf_score.columns[:-1]
ko_name = ['누적수익율Z점수', '정보비율Z점수', '샤프지수Z점수', '상관관계Z점수', '트래킹에러Z점수', '최대낙폭Z점수', '변동성Z점수']

# subplot 설정
num_metrics = len(metrics)
fig, axes = plt.subplots(nrows=(num_metrics + 1) // 4, ncols=4, figsize=(12, 3 * ((num_metrics + 1) // 4)))
axes = axes.flatten()  

# 색상 팔레트 정의
palette = sns.color_palette("husl", n_colors=len(etf_score['cluster'].unique()))

for i, metric in enumerate(metrics):
    sns.boxplot(x='cluster', y=metric, data=etf_score, palette=palette, ax=axes[i])
    axes[i].set_title(f'Boxplot of {ko_name[i]}')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel(metric)

# 남은 subplot 비우기
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

- 군집별 해석은 다음과 같습니다.

    - 군집 0 : 이 군집은 중간 정도의 수익률을 보여주지만 변동성(Z점수)과 최대 낙폭(Z점수)이 커 투자 위험이 클 수 있습니다.   
        그리고 상관관계(Z점수)이 높은 것으로 나타나, 비교적 다른 ETF와 연관성이 높다는 특징을 보입니다.


    - 군집 1 : 이 군집은 누적수익율(Z점수)과 정보비율(Z점수)이 높아 높은 수익률을 보여주며, 최대 낙폭(Z점수)과 변동성(Z점수)가 높지 않은 편이라 상대적으로 안정적인 투자 대상일 가능성이 큽니다.   
    또한 상관관계(Z점수)는 낮아 다른 ETF들과의 연관성은 적으며, 트래킹 에러(Z점수)가 높은 편이라는 점에서 벤치마크와의 차이가 클 수 있습니다.


    - 군집 2 : 이 군집은 샤프지수(Z점수)와 누적수익율(Z점수)이 낮아 낮은 수익률을 기록하고 있지만, 최대 낙폭(Z점수)과 변동성(Z점수)도 낮아 안정적입니다.  
    그리고 트래킹 에러(Z점수)는 매우 높아 벤치마크와의 차이가 클 수 있습니다.


- 최종 군집 해석은 아래와 같습니다.

  군집 0 : 투자를 권장하지 않는 ETF 군집(중간 정도의 성과와 높은 변동성을 지닌 군집)

  군집 1 : 일반적으로 투자를 권장하는 ETF 군집(높은 성과와 높지 않은 변동성을 지닌 군집)

  군집 2 : 안정성이 우선인 투자자를 위한 ETF 군집(낮은 성과와 낮은 변동성을 가진 군집)

## 각 군집 별 대표 ETF들을 도출

- 각 군집의 중심을 구하고, 그 중심에 가장 가까운 순으로 그 군집을 대표하는 ETF라고 판단했습니다.
- 아래는 각 군집별로 대표 ETF 3개씩 구한 결과입니다.

In [ ]:
# 군집 별 대표 ETF 구하기

# 클러스터 중심 좌표 가져오기
centroids = kmeans.cluster_centers_

# 각 군집별로 중심에서 가까운 데이터 3개씩 구하기
closest_data_per_cluster = {}
for i in range(optimal_n_clusters):
    # 군집 i에 속한 데이터의 인덱스를 찾기
    cluster_indices = np.where(labels == i)[0]
    cluster_data = etf_score.iloc[cluster_indices, :-1]
    
    # 클러스터 중심과의 거리 계산
    distances = np.linalg.norm(cluster_data - centroids[i], axis=1)
    
    # 거리 기준으로 가까운 3개의 데이터 선택
    closest_indices = cluster_indices[np.argsort(distances)[:3]]
    
    # 결과 저장
    closest_data_per_cluster[i] = closest_indices

# 각 군집별로 가까운 3개 데이터 출력
for cluster, indices in closest_data_per_cluster.items():
    #print(f"Cluster {cluster}: Closest data indices: {indices}")
    print(f"Cluster {cluster}의 대표 ETF 3개 : {list(etf_score.iloc[indices].index)}")

# 2. 생성형 AI를 이용해 ETF 구성종목들의 설명으로 요약

- ETF들 중 군집 1의 대표 ETF인 QQQ로 예시를 들어 구현했습니다.

In [ ]:
# ETF구성종목정보, 해외종목정보 테이블 데이터 불러오기
import pandas as pd
ETF_HOLDINGS = pd.read_csv('./data/NH_CONTEST_DATA_ETF_HOLDINGS.csv', encoding='cp949')
NW_FC_STK_IEM_IFO = pd.read_csv('./data/NH_CONTEST_NW_FC_STK_IEM_IFO.csv', encoding='cp949')

In [ ]:
# ETF구성종목정보 테이블에서 대상 ETF(QQQ)의 구성종목, 구성 비율을 추출
ETF_HOLDINGS_sample = ETF_HOLDINGS[ETF_HOLDINGS['etf_tck_cd'] == 'QQQ'][['etf_tck_cd', 'tck_iem_cd', 'wht_pct']]
ETF_HOLDINGS_sample

In [ ]:
# 해외종목정보 테이블에서 종목코드, 영문사업개요내용을 추출
NW_FC_STK_IEM_IFO_sample = NW_FC_STK_IEM_IFO[['tck_iem_cd', 'eng_utk_otl_cts']]
NW_FC_STK_IEM_IFO_sample

- microsoft azure openai의 gpt 토큰 제한으로 ETF의 전 종목을 입력시키는 것이 불가능했습니다.

    - 유료버전의 azure openai를 사용한다면 더 높은 버전의 gpt 사용으로 더 많은 입력을 받을 수 있음을 기대할 수 있습니다.
- 상위 5종목만 입력시켜서 이 기능이 진행됨을 보여드리겠습니다.


In [ ]:
# 대상 ETF(QQQ)의 구성종목의 영문사업개요내용을 확인
etf_describe = pd.merge(ETF_HOLDINGS_sample, NW_FC_STK_IEM_IFO_sample, on='tck_iem_cd')
etf_describe.sort_values(by='wht_pct', ascending=False)

In [ ]:
# 상위 5개만 선택 후 etf 코드는 필요없기 때문에 제거
etf_describe_top5 = etf_describe.head(5)
etf_describe_top5.drop('etf_tck_cd', axis=1, inplace=True)
etf_describe_top5

In [ ]:
# 데이터프레임을 microsoft azure openai에 입력할 수 있는 형태로 변환 필요

# 데이터프레임을 딕셔너리로 변환
df_dict = etf_describe_top5.to_dict(orient='records')

# 닉셔너리를 문자열로 변환
df_dict_string = '\n'.join([str(record) for record in df_dict])

# 결과 출력
print(df_dict_string)

## 생성형 AI를 이용해 큐레이션하고자 하는 ETF의 구성 종목들의 설명들을 요약

- Microsoft Azure openai를 이용해 생성형 AI(gpt3.5-turbo)를 사용했습니다.

- ETF 구성 종목들의 설명(영문사업개요내용)을 생성형 AI에 입력합니다.

    - 입력받은 내용을 요약하도록 요청합니다.

    - 추후 키워드의 중요도를 구하기 위해 영문 요약본도 같이 도출합니다.

In [ ]:
import os
import requests
import base64

# Configuration
API_KEY = "YOUR_API_KEY"
# IMAGE_PATH = "YOUR_IMAGE_PATH"
# encoded_image = base64.b64encode(open(IMAGE_PATH, 'rb').read()).decode('ascii')
headers = {
    "Content-Type": "application/json",
    "api-key": API_KEY,
}

# Payload for the request
payload = {
  "messages": [
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "나는 너한테 dictionary 형태로 ETF에 대해 어떤 주식 종목들이 구성하고 있는지에 대한 정보를 줄거야. \n 종목들의 설명들을 종합해서 ETF에 대한 구체적인 주요 정보들을 영어로 요약해주고 그 내용 그대로 한글도 요약해줘(단,  영어 문장과 한글 문장은 줄 바꿈 해줘).\n 그리고 요약본 길이는 3 문장 이내로 하고, 내용에 종목명을 언급하지말아줘. \n  제공하는 정보는 ETF를 구성하는 주식 종목 코드(tck_iem_cd), ETF에서의 주식 구성 비율(wht_pct), 주식 설명(eng_utk_otl_cts)을 줄거야. \n그 요약본은 ETF 큐레이션에 사용될거야 큐레이션에 필요한 정보들을 풍부하지만 간결하게 담아줘."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": df_dict_string
        }
      ]
    }
  ],
  "temperature": 0.7,
  "top_p": 0.95,
  "max_tokens": 800
}

ENDPOINT = "https://<resource-name>.openai.azure.com/openai/deployments/<deployment-name>/chat/completions?api-version=2024-02-15-preview"

# Send request
try:
    response = requests.post(ENDPOINT, headers=headers, json=payload)
    response.raise_for_status()  # Will raise an HTTPError if the HTTP request returned an unsuccessful status code
except requests.RequestException as e:
    raise SystemExit(f"Failed to make the request. Error: {e}")

# Handle the response as needed (e.g., print or process)
print(response.json()['choices'][0]['message']['content'])

In [ ]:
en_summation = response.json()['choices'][0]['message']['content'].split('\n')[0]
en_summation

In [ ]:
ko_summation = response.json()['choices'][0]['message']['content'].split('\n')[2]
ko_summation

# 3. ETF의 요약글 속 단어들의 중요도 큐레이션

## ETF 구성 종목들의 설명 속 키워드와 수익률 간의 관계 파악

- 해외종목정보 테이블의 영문사업개요내용(eng_utk_otl_cts)을 종목들의 설명으로 간주합니다.

In [ ]:
# 해외종목정보 테이블 불러오기
NW_FC_STK_IEM_IFO = pd.read_csv('./data/NH_CONTEST_NW_FC_STK_IEM_IFO.csv', encoding='cp949')
NW_FC_STK_IEM_IFO.head()

In [ ]:
# ETF인 경우 제외 추출
NW_FC_STK_IEM_IFO = NW_FC_STK_IEM_IFO[NW_FC_STK_IEM_IFO['stk_etf_dit_cd'] != 'ETF']

# 영문사업개요내용(ENG_UTK_OTL_CTS)의 길이 열 추가
NW_FC_STK_IEM_IFO.loc[:, 'eng_utk_otl_cts_len'] = NW_FC_STK_IEM_IFO['eng_utk_otl_cts'].apply(lambda x: len(x))

In [ ]:
# 영문사업개요내용이 없는 행 제거 
NW_FC_STK_IEM_IFO = NW_FC_STK_IEM_IFO[NW_FC_STK_IEM_IFO['eng_utk_otl_cts_len'] > 1]
NW_FC_STK_IEM_IFO.head()

- 영문사업개요내용 열에 대한 텍스트 전처리를 진행합니다.

In [ ]:
# 전처리 함수 정의
def preprocess_text(text):
    # 소문자 변환
    text = text.lower()
    # 영어 외의 다른 문자 제거
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 구두점 제거
    text = re.sub(r'[^\w\s]', '', text)
    # 불용어 제거
    stop_words = set(stopwords.words('english'))
    # 종목 코드 불용어에 추가
    stop_words = stop_words.union(set(NW_FC_STK_IEM_IFO['tck_iem_cd'].apply(lambda x: x.lower()).values))
    # 종목명 불용어에 추가
    stop_words = stop_words.union(set(NW_FC_STK_IEM_IFO['fc_sec_eng_nm'].apply(lambda x: x.lower()).str.split().sum()))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    # 어간 추출 
    stemmer = PorterStemmer()
    text = ' '.join([stemmer.stem(word) for word in text.split()])
    return text

# 영문사업개요내용 열 전처리
NW_FC_STK_IEM_IFO['eng_utk_otl_cts_pro'] = NW_FC_STK_IEM_IFO['eng_utk_otl_cts'].apply(preprocess_text)
NW_FC_STK_IEM_IFO['eng_utk_otl_cts_pro']

- ETF의 구성종목들의 설명과 수익률 간의 관계를 파악합니다.

    - 각 종목의 최근 5영업일 간의 평균 수익률을 사용합니다.

In [ ]:
# 주식일별정보 테이블 불러오기
STK_DD_IFO = pd.read_csv('./data/NH_CONTEST_NHDATA_STK_DD_IFO.csv', encoding='cp949')

# 종목 코드의 오른쪽 공백 제거
STK_DD_IFO['tck_iem_cd'] = STK_DD_IFO['tck_iem_cd'].str.rstrip()

# 날짜 기준으로 정렬
STK_DD_IFO.sort_values(by=['tck_iem_cd', 'bse_dt'], inplace=True)
STK_DD_IFO.reset_index(drop=True, inplace=True)
STK_DD_IFO

In [ ]:
# 주식일별정보 테이블에서 최근 5영업일 간의 평균 수익률 계산
earn_df = STK_DD_IFO[STK_DD_IFO['bse_dt'] >= STK_DD_IFO['bse_dt'].unique()[-5]].groupby('tck_iem_cd').mean()[['tco_avg_pft_rt']]
earn_df

In [ ]:
# 해외종목정보 테이블에서 종목코드, 영문사업개요내용을 추출
NW_FC_STK_IEM_IFO_sample = NW_FC_STK_IEM_IFO[['tck_iem_cd', 'eng_utk_otl_cts_pro']]

In [ ]:
# 영문사업개요내용과 5영업일 간의 평규 수익률을 조인
earn_description_df = pd.merge(NW_FC_STK_IEM_IFO_sample, earn_df, left_on='tck_iem_cd', right_on= earn_df.index)
earn_description_df

- 의미있는 키워드 도출에 적합한 TF-IDF를 영문사업개요내용에 적용합니다.

In [ ]:
# TF-IDF 벡터라이저 생성
tfidf_vectorizer = TfidfVectorizer(max_features=100)

# 전처리된 영문사업개요내용 열에 TF-IDF 적용
tfidf_matrix = tfidf_vectorizer.fit_transform(earn_description_df['eng_utk_otl_cts_pro'])

# 결과를 데이터프레임으로 변환
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df

In [ ]:
# tfidf_df에 종목코드, 평균 수익률 추가
tfidf_df['tck_iem_cd'] = earn_description_df['tck_iem_cd']
tfidf_df['tco_avg_pft_rt'] = earn_description_df['tco_avg_pft_rt']
tfidf_df

- TF-IDF 결과와 수익률 간의 관계를 보기 위해 회귀모델(XGBoost 사용)을 구축합니다.

In [ ]:
# tfidf 결과로 5영업일 간의 평균 수익률을 예측하는 회귀 모델 생성
X = tfidf_df.drop(['tck_iem_cd', 'tco_avg_pft_rt'], axis=1)
y = tfidf_df['tco_avg_pft_rt']

# XGBoost 회귀 모델 생성 및 훈련
xgb_reg = XGBRegressor(random_state=42)
xgb_reg.fit(X, y)

## XAI를 통한 설명 속 키워드의 중요도 산정 모델 구축

- XAI 중 SHAP를 이용해, 설명과 수익률 관의 모델이 중요하게 생각하는 키워드를 도출

In [ ]:
# SHAP 모델 생성 후 shap value 계산
explainer = shap.Explainer(xgb_reg)
shap_values = explainer(X)

In [ ]:
# 영문사업개요내용 키워드의 중요도 순으로 내림차순한 그래프
shap.summary_plot(shap_values, X, plot_type="bar")

- ETF의 구성종목들 각각에 대해서, 수익률에 긍정, 부정을 미친 키워드 또한 투자자에게 제공합니다.

In [ ]:
# ETF를 구성하는 비율이 가장 높은 종목들의 인덱스 추출(주식일별정보 테이블에서 DLTR가 없어서, 상위 5개 중 4개만 구현 )
top_index = tfidf_df[tfidf_df['tck_iem_cd'].isin(etf_describe_top5['tck_iem_cd'])].index
shap_values = explainer(X)
# 각 종목의 영문사업개요내용이 수익률에 미치는 긍부정 중요도 그래프 생성
shap_values_list = [shap_values[top_index[0]], shap_values[top_index[1]], shap_values[top_index[2]], shap_values[top_index[3]]]  # 예시로 4개의 shap_values를 넣음
titles = [f"{tfidf_df.loc[top_index[0], 'tck_iem_cd']}", f"{tfidf_df.loc[top_index[1], 'tck_iem_cd']}", f"{tfidf_df.loc[top_index[2], 'tck_iem_cd']}", f"{tfidf_df.loc[top_index[3], 'tck_iem_cd']}"]  # 각 그래프의 제목 리스트

fig, axes = plt.subplots(2, 2, figsize=(15, 10)) 
for i, shap_values in enumerate(shap_values_list):
    row = i // 2 
    col = i % 2   
    shap.plots.bar(shap_values, show=False, ax=axes[row, col]) 
    axes[row, col].set_title(titles[i]) 
plt.tight_layout()
plt.show()

## ETF의 요약글 속 단어들의 중요도 큐레이션

In [ ]:
# 단어별 중요도 데이터프레임 생성
shap_values = explainer.shap_values(X)
shap_abs_mean = np.mean(np.abs(shap_values), axis=0)

importance_df = pd.DataFrame({
    'feature': X.columns, 
    'importance': shap_abs_mean  
})
importance_df

In [ ]:
# 영어 요약본 속 단어들의 중요도 도출
print(en_summation)
importance_df[importance_df['feature'].isin(en_summation.split())]

- 위의 결과에 따르면 아래 한글 요약본에서 콘텐츠, 데이터, 플렛폼 키워드가 수익률에 유의미한 영향을 미쳤음을 알 수 있습니다.
- 이 정보를 최종적으로 투자자에게 큐레이션합니다.

In [ ]:

ko_summation